# LAB 08 — SUBWORD TOKENIZATION, SPECIAL TOKENS, ATTENTION MASK & MLM LOSS
**Môn học**: Xử lý ngôn ngữ tự nhiên và ứng dụng (MAT3561) — Học kỳ I - 2026  
**Học viên thực hiện**: Vũ Tiến Đạt — **MSSV**: `23000111`  
**Giảng viên lý thuyết**: TS. Lê Hồng Phương  
**Giảng viên thực hành**: ThS. Phạm Ngọc Hải

---
Notebook này triển khai thực nghiệm chi tiết các **Mục 4, 5, 6, 7, 8, 9, 10 và 12.1** trong `W8.pdf`.

## 1. Khởi tạo Môi trường & Cố định Random Seed (Mục 23)
Thiết lập môi trường tính toán và import module `pretrained_classification.py`.

In [1]:
import random
import math
import numpy as np
from pretrained_classification import (
    BertWordPieceTokenizer,
    compute_mlm_loss,
    stable_softmax,
)

random.seed(42)
np.random.seed(42)
tokenizer = BertWordPieceTokenizer()
print("Khởi tạo BertWordPieceTokenizer thành công! Vocab size:", tokenizer.vocab_size)

Khởi tạo BertWordPieceTokenizer thành công! Vocab size: 30522


## 5. Subword Tokenization & 6. Special Tokens
- Kiểm tra cơ chế phân tách từ phức/từ hiếm thành các mảnh subword (`unbelievable -> un + ##believe + ##able`).
- Kiểm tra vai trò của 5 Special Tokens chuẩn BERT: `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`, `[UNK]`.

In [2]:
sample_words = ["movie", "great", "unbelievable", "unhappily", "cinematography", "rewatchable"]
print("=== MỤC 5: MINH HỌA SUBWORD TOKENIZATION (WORDPIECE) ===")
for w in sample_words:
    subwords = tokenizer.tokenize(w)
    print(f"  Word: {w:<16} -> Subword tokens: {subwords} (Số subwords: {len(subwords)})")

print("\n=== MỤC 6: BẢNG SPECIAL TOKENS TRONG BERT-STYLE MODELS ===")
special_roles = {
    "[CLS]": "Đại diện cho toàn bộ sequence ở đầu chuỗi trong classification setup",
    "[SEP]": "Phân tách ranh giới câu hoặc đánh dấu kết thúc sequence",
    "[PAD]": "Đệm chiều dài để gom các câu có độ dài khác nhau vào cùng 1 batch",
    "[MASK]": "Che token mục tiêu trong quá trình tiền huấn luyện MLM",
    "[UNK]": "Đại diện cho ký tự/token lạ ngoài khả năng phân tách của từ điển",
}
for tok, token_id in tokenizer.SPECIAL_TOKENS.items():
    print(f"  {tok:<8} (ID = {token_id:<3}) : {special_roles[tok]}")

=== MỤC 5: MINH HỌA SUBWORD TOKENIZATION (WORDPIECE) ===
  Word: movie            -> Subword tokens: ['movie'] (Số subwords: 1)
  Word: great            -> Subword tokens: ['great'] (Số subwords: 1)
  Word: unbelievable     -> Subword tokens: ['un', '##believe', '##able'] (Số subwords: 3)
  Word: unhappily        -> Subword tokens: ['un', '##happi', '##ly'] (Số subwords: 3)
  Word: cinematography   -> Subword tokens: ['cinema', '##to', '##graphy'] (Số subwords: 3)
  Word: rewatchable      -> Subword tokens: ['re', '##watch', '##able'] (Số subwords: 3)

=== MỤC 6: BẢNG SPECIAL TOKENS TRONG BERT-STYLE MODELS ===
  [PAD]    (ID = 0  ) : Đệm chiều dài để gom các câu có độ dài khác nhau vào cùng 1 batch
  [UNK]    (ID = 100) : Đại diện cho ký tự/token lạ ngoài khả năng phân tách của từ điển
  [CLS]    (ID = 101) : Đại diện cho toàn bộ sequence ở đầu chuỗi trong classification setup
  [SEP]    (ID = 102) : Phân tách ranh giới câu hoặc đánh dấu kết thúc sequence
  [MASK]   (ID = 103) : Che to

## 7. Attention Mask
Kiểm chứng chuỗi ví dụ trong đề bài:
`[CLS] the cat eats fish [PAD] [PAD]` $\longrightarrow$ `attention_mask = [1, 1, 1, 1, 1, 0, 0]`
và chứng minh các vị trí `[PAD]` có trọng số Self-Attention bằng `0.0000` sau khi áp dụng mask.

In [3]:
seq_tokens = ["[CLS]", "the", "cat", "eats", "fish", "[PAD]", "[PAD]"]
attn_mask = [1 if t != "[PAD]" else 0 for t in seq_tokens]
print("=== MỤC 7: ATTENTION MASK ===")
print("Sequence      :", seq_tokens)
print("Attention mask:", attn_mask)

raw_scores = np.array([0.8, 1.2, 1.5, 1.1, 0.9, 2.0, 1.7])
masked_scores = np.where(np.array(attn_mask) == 1, raw_scores, -1e9)
attn_weights_unmasked = stable_softmax(raw_scores)
attn_weights_masked = stable_softmax(masked_scores)
print("\nTrọng số Attention nếu KHÔNG có mask (bị nhiễu bởi [PAD]):")
print(" ", np.round(attn_weights_unmasked, 4))
print("Trọng số Attention khi CÓ attention_mask ([PAD] bị triệt tiêu về 0):")
print(" ", np.round(attn_weights_masked, 4))

=== MỤC 7: ATTENTION MASK ===
Sequence      : ['[CLS]', 'the', 'cat', 'eats', 'fish', '[PAD]', '[PAD]']
Attention mask: [1, 1, 1, 1, 1, 0, 0]

Trọng số Attention nếu KHÔNG có mask (bị nhiễu bởi [PAD]):
  [0.0785 0.1171 0.1581 0.106  0.0867 0.2606 0.1931]
Trọng số Attention khi CÓ attention_mask ([PAD] bị triệt tiêu về 0):
  [0.1437 0.2143 0.2893 0.1939 0.1588 0.     0.    ]


## 8. Masked Language Modeling (MLM) & 9. Calculation Exercise 1 — MLM Loss
Cho câu `"The cat [MASK] fish."` với phân phối xác suất dự đoán tại vị trí `[MASK]`:
- `cat: 0.10`, `dog: 0.05`, `eats: 0.70`, `fish: 0.15` (Token đúng: `eats`).
- Tính $L = -\log(0.70)$ và khảo sát sự thay đổi khi $P(\text{eats}) = 0.95$ và $P(\text{eats}) = 0.01$.

In [4]:
mlm_preds = {"cat": 0.10, "dog": 0.05, "eats": 0.70, "fish": 0.15}
L_070_ln = compute_mlm_loss(mlm_preds, "eats")
L_095_ln = compute_mlm_loss({"eats": 0.95}, "eats")
L_001_ln = compute_mlm_loss({"eats": 0.01}, "eats")

print("=== MỤC 9: CALCULATION EXERCISE 1 — MLM CROSS-ENTROPY LOSS ===")
print(f"1. Với P('eats') = 0.70 : L = -ln(0.70) = {L_070_ln:.4f} (hoặc -log10(0.70) = {-math.log10(0.70):.4f})")
print(f"2. Nếu P('eats') = 0.95 : L = -ln(0.95) = {L_095_ln:.4f} -> Loss giảm mạnh gần về 0 (giảm {L_070_ln/L_095_ln:.1f} lần)")
print(f"3. Nếu P('eats') = 0.01 : L = -ln(0.01) = {L_001_ln:.4f} -> Loss tăng vọt gấp {L_001_ln/L_070_ln:.1f} lần để phạt nặng dự đoán sai")

=== MỤC 9: CALCULATION EXERCISE 1 — MLM CROSS-ENTROPY LOSS ===
1. Với P('eats') = 0.70 : L = -ln(0.70) = 0.3567 (hoặc -log10(0.70) = 0.1549)
2. Nếu P('eats') = 0.95 : L = -ln(0.95) = 0.0513 -> Loss giảm mạnh gần về 0 (giảm 7.0 lần)
3. Nếu P('eats') = 0.01 : L = -ln(0.01) = 4.6052 -> Loss tăng vọt gấp 12.9 lần để phạt nặng dự đoán sai


## 10. Calculation Exercise 2 — Tokenization
Thực hiện đầy đủ 6 bước cho câu: `"The cats are playing."`
1. Tokenize bằng tokenizer được cung cấp.
2. Ghi token sequence.
3. Ghi token IDs.
4. Xác định special tokens.
5. Xác định attention mask.
6. So sánh số lượng: `words`, `tokens`, `subword tokens`.

In [5]:
sentence = "The cats are playing."
encoded = tokenizer.encode_single(sentence)
encoded_padded = tokenizer.encode_single(sentence, max_length=10, padding=True)

raw_words = sentence.split()
subword_body = tokenizer.tokenize(sentence)
morph_subwords = ["the", "cat", "##s", "are", "play", "##ing", "."]

print("=== MỤC 10: CALCULATION EXERCISE 2 — TOKENIZATION ===")
print(f"Câu gốc                        : {sentence!r}")
print(f"1 & 2. Token sequence          : {encoded['tokens']}")
print(f"3. Token IDs                   : {encoded['input_ids']}")
print(f"4. Special tokens              : ['[CLS]' (101), '[SEP]' (102)]")
print(f"5. Attention mask (no pad)     : {encoded['attention_mask']}")
print(f"   Attention mask (pad len=10) : {encoded_padded['attention_mask']}")
print("\n6. So sánh số lượng Words vs Tokens vs Subword Tokens:")
print(f"   - Words (tách khoảng trắng) : {len(raw_words)} words -> {raw_words}")
print(f"   - Subword tokens (thân câu) : {len(subword_body)} tokens (bert-base-uncased) -> {subword_body}")
print(f"                                 (Nếu tách hình thái gốc + hậu tố: {len(morph_subwords)} subwords -> {morph_subwords})")
print(f"   - Total tokens (vào model)  : {len(encoded['tokens'])} tokens (bao gồm [CLS], [SEP] và dấu chấm câu '.')")

=== MỤC 10: CALCULATION EXERCISE 2 — TOKENIZATION ===
Câu gốc                        : 'The cats are playing.'
1 & 2. Token sequence          : ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
3. Token IDs                   : [101, 1996, 8870, 2024, 2652, 1012, 102]
4. Special tokens              : ['[CLS]' (101), '[SEP]' (102)]
5. Attention mask (no pad)     : [1, 1, 1, 1, 1, 1, 1]
   Attention mask (pad len=10) : [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]

6. So sánh số lượng Words vs Tokens vs Subword Tokens:
   - Words (tách khoảng trắng) : 4 words -> ['The', 'cats', 'are', 'playing.']
   - Subword tokens (thân câu) : 5 tokens (bert-base-uncased) -> ['the', 'cats', 'are', 'playing', '.']
                                 (Nếu tách hình thái gốc + hậu tố: 7 subwords -> ['the', 'cat', '##s', 'are', 'play', '##ing', '.'])
   - Total tokens (vào model)  : 7 tokens (bao gồm [CLS], [SEP] và dấu chấm câu '.')


## 12.1. Core Implementation — Kiểm tra `tokens.keys()`, `input_ids`, `attention_mask`
Chạy lệnh chuẩn:
```python
tokens = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
```
và kiểm tra `tokens.keys()`, `input_ids`, `attention_mask`.

In [6]:
text_batch = [
    "The cats are playing.",
    "This movie is an unbelievable masterpiece!"
]
tokens = tokenizer(
    text_batch,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("=== MỤC 12.1: KIỂM TRA TOKENS.KEYS() VÀ GIẢI THÍCH CÁC TRƯỜNG ===")
print("tokens.keys() ->", list(tokens.keys()))
print("\n1. input_ids (ID số nguyên của từng token trong từ điển, 0 là [PAD]):")
print(tokens["input_ids"])
print("\n2. attention_mask (1 = token hợp lệ cần chú ý, 0 = [PAD] bị bỏ qua):")
print(tokens["attention_mask"])
print("\n3. token_type_ids (Segment IDs: 0 cho câu đơn):")
print(tokens["token_type_ids"])

=== MỤC 12.1: KIỂM TRA TOKENS.KEYS() VÀ GIẢI THÍCH CÁC TRƯỜNG ===
tokens.keys() -> ['input_ids', 'token_type_ids', 'attention_mask']

1. input_ids (ID số nguyên của từng token trong từ điển, 0 là [PAD]):
[[  101  1996  8870  2024  2652  1012   102     0     0     0     0]
 [  101  2023  3185  2003  2019  4895 18332  3085 17743   999   102]]

2. attention_mask (1 = token hợp lệ cần chú ý, 0 = [PAD] bị bỏ qua):
[[1 1 1 1 1 1 1 0 0 0 0]
 [1 1 1 1 1 1 1 1 1 1 1]]

3. token_type_ids (Segment IDs: 0 cho câu đơn):
[[0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]]
